CELL 1


In [1]:
!pip install xgboost mlflow -q

from google.colab import files
uploaded = files.upload()  # upload emi_prediction_dataset.csv when prompted

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Saving emi_prediction_dataset.csv to emi_prediction_dataset.csv


cell 1b - verify GPU + XGBoost version before training

In [2]:
import xgboost as xgb
print("XGBoost version:", xgb.__version__)
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv


XGBoost version: 3.4.0
name, memory.used [MiB], memory.total [MiB], utilization.gpu [%]
Tesla T4, 0 MiB, 15360 MiB, 0 %


cell 2


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('emi_prediction_dataset.csv', low_memory=False)
print("Loaded:", df.shape)

# downcast to shrink memory footprint (404k rows adds up fast)
for c in df.select_dtypes(include='float64').columns:
    df[c] = df[c].astype('float32')
for c in df.select_dtypes(include='int64').columns:
    df[c] = df[c].astype('int32')
print("Memory after downcast:", round(df.memory_usage(deep=True).sum() / 1e6, 1), "MB")

def clean_numeric_string(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r'(\.0)+$', '', regex=True),
        errors='coerce'
    )

for col in ['age', 'monthly_salary', 'bank_balance']:
    df[col] = clean_numeric_string(df[col])

df['gender'] = df['gender'].astype(str).str.strip().str.lower().map({
    'male': 'Male', 'm': 'Male', 'female': 'Female', 'f': 'Female'
})

for col in ['monthly_rent', 'credit_score', 'bank_balance', 'emergency_fund']:
    df[col] = df[col].fillna(df[col].median())
df['education'] = df['education'].fillna(df['education'].mode()[0])

print("Missing after cleanup:", df.isna().sum().sum())

Loaded: (404800, 27)
Memory after downcast: 298.5 MB
Missing after cleanup: 0


cell 3

In [4]:
total_expenses = (df['monthly_rent'] + df['school_fees'] + df['college_fees'] +
                   df['travel_expenses'] + df['groceries_utilities'] + df['other_monthly_expenses'])

df['total_monthly_expenses'] = total_expenses
df['debt_to_income'] = (df['current_emi_amount'] + total_expenses) / df['monthly_salary'].replace(0, np.nan)
df['emi_burden_ratio'] = df['current_emi_amount'] / df['monthly_salary'].replace(0, np.nan)
df['expense_ratio'] = total_expenses / df['monthly_salary'].replace(0, np.nan)
df['savings_ratio'] = df['emergency_fund'] / df['monthly_salary'].replace(0, np.nan)
df['loan_to_income'] = df['requested_amount'] / (df['monthly_salary'] * 12).replace(0, np.nan)
df['financial_buffer'] = df['monthly_salary'] - df['current_emi_amount'] - total_expenses
df['dependents_per_income'] = df['dependents'] / (df['monthly_salary'] / 10000).replace(0, np.nan)

ratio_cols = ['debt_to_income','emi_burden_ratio','expense_ratio','savings_ratio','loan_to_income','dependents_per_income']
for c in ratio_cols:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan)
    df[c] = df[c].fillna(df[c].median())

print("Feature engineering done:", df.shape)

Feature engineering done: (404800, 35)


cell 4

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

cat_cols = ['gender','marital_status','education','employment_type','company_type',
            'house_type','existing_loans','emi_scenario']
num_cols = [c for c in df.columns if c not in cat_cols + ['emi_eligibility','max_monthly_emi']]

df_enc = df.copy()
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df_enc[c] = le.fit_transform(df_enc[c])
    encoders[c] = le

feature_cols = num_cols + cat_cols
X = df_enc[feature_cols]

import gc
del df
gc.collect()


80

cell 5

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report
import warnings; warnings.filterwarnings('ignore')

y_clf = df_enc['emi_eligibility']
le_target = LabelEncoder()
y_clf_enc = le_target.fit_transform(y_clf)
print("Class mapping:", dict(zip(le_target.classes_, range(len(le_target.classes_)))))

Xtr, Xte, ytr, yte = train_test_split(X, y_clf_enc, test_size=0.2, random_state=42, stratify=y_clf_enc)
scaler_c = StandardScaler()
Xtr_s, Xte_s = scaler_c.fit_transform(Xtr), scaler_c.transform(Xte)

clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, random_state=42, eval_metric='mlogloss',
                              tree_method='hist', device='cuda', n_jobs=-1)
}

clf_results, clf_fitted = {}, {}
for name, model in clf_models.items():
    if name == 'Logistic Regression':
        model.fit(Xtr_s, ytr); preds = model.predict(Xte_s); proba = model.predict_proba(Xte_s)
    elif name == 'XGBoost':
        sw = compute_sample_weight('balanced', ytr)
        model.fit(Xtr, ytr, sample_weight=sw); preds = model.predict(Xte); proba = model.predict_proba(Xte)
    else:
        model.fit(Xtr, ytr); preds = model.predict(Xte); proba = model.predict_proba(Xte)

    acc = accuracy_score(yte, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(yte, preds, average='macro')
    auc = roc_auc_score(yte, proba, multi_class='ovr')
    clf_results[name] = {'accuracy': acc, 'f1_macro': f1, 'roc_auc': auc}
    clf_fitted[name] = model
    print(f"\n--- {name} ---\nAccuracy: {acc:.4f} | Macro-F1: {f1:.4f} | ROC-AUC: {auc:.4f}")
    print(classification_report(yte, preds, target_names=le_target.classes_))

print(pd.DataFrame(clf_results).T.round(4))

Class mapping: {'Eligible': 0, 'High_Risk': 1, 'Not_Eligible': 2}

--- Logistic Regression ---
Accuracy: 0.7897 | Macro-F1: 0.6385 | ROC-AUC: 0.9256
              precision    recall  f1-score   support

    Eligible       0.83      0.76      0.79     14889
   High_Risk       0.14      0.67      0.24      3497
Not_Eligible       0.98      0.80      0.88     62574

    accuracy                           0.79     80960
   macro avg       0.65      0.74      0.64     80960
weighted avg       0.92      0.79      0.84     80960


--- Random Forest ---
Accuracy: 0.9439 | Macro-F1: 0.6512 | ROC-AUC: 0.9785
              precision    recall  f1-score   support

    Eligible       0.91      0.94      0.93     14889
   High_Risk       0.77      0.03      0.05      3497
Not_Eligible       0.95      1.00      0.97     62574

    accuracy                           0.94     80960
   macro avg       0.88      0.66      0.65     80960
weighted avg       0.94      0.94      0.92     80960


--- XGBoost

In [7]:
import gc
del Xtr_s, Xte_s, proba
gc.collect()


74

cell 6

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_reg = df_enc['max_monthly_emi']
Xtr, Xte, ytr, yte = train_test_split(X, y_reg, test_size=0.2, random_state=42)
scaler_r = StandardScaler()
Xtr_s, Xte_s = scaler_r.fit_transform(Xtr), scaler_r.transform(Xte)

reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, random_state=42,
                             tree_method='hist', device='cuda', n_jobs=-1)
}

reg_results, reg_fitted = {}, {}
for name, model in reg_models.items():
    if name == 'Linear Regression':
        model.fit(Xtr_s, ytr); preds = model.predict(Xte_s)
    else:
        model.fit(Xtr, ytr); preds = model.predict(Xte)

    rmse = np.sqrt(mean_squared_error(yte, preds))
    mae = mean_absolute_error(yte, preds)
    r2 = r2_score(yte, preds)
    reg_results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    reg_fitted[name] = model
    print(f"--- {name} ---\nRMSE: {rmse:.2f} | MAE: {mae:.2f} | R2: {r2:.4f}\n")

print(pd.DataFrame(reg_results).T.round(4))

--- Linear Regression ---
RMSE: 4110.18 | MAE: 2945.86 | R2: 0.7138



In [ ]:
import gc
del Xtr, Xte, Xtr_s, Xte_s, preds
gc.collect()


cell 7 save the best models #such as regressor,classifier

In [ ]:
import joblib
joblib.dump(clf_fitted['XGBoost'], 'best_classifier.pkl')
joblib.dump(reg_fitted['XGBoost'], 'best_regressor.pkl')
joblib.dump(scaler_c, 'scaler_classification.pkl')
joblib.dump(scaler_r, 'scaler_regression.pkl')
joblib.dump(encoders, 'label_encoders.pkl')
joblib.dump(le_target, 'target_encoder.pkl')
files.download('best_classifier.pkl')
files.download('best_regressor.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('scaler_classification.pkl')
files.download('scaler_regression.pkl')
files.download('label_encoders.pkl')
files.download('target_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

cell 8 - MLflow experiment tracking

In [ ]:
import mlflow

mlflow.set_experiment("EMIPredict-AI")

# NOTE: model artifacts are NOT logged here (mlflow.sklearn.log_model /
# mlflow.xgboost.log_model) because serializing the Random Forest (200 trees)
# and XGBoost (300 trees) models in-memory alongside everything already
# loaded is what crashed the Colab session on RAM. Params + metrics are all
# that's needed for experiment tracking; your actual model files are already
# saved separately via joblib.dump() in Cell 7 / files.download().

# ---------- Classification runs ----------
for name, model in clf_fitted.items():
    with mlflow.start_run(run_name=f"clf_{name.replace(' ', '_')}"):
        mlflow.set_tag("task", "classification")
        mlflow.log_param("model_type", name)
        mlflow.log_param("n_features", X.shape[1])
        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)
        if hasattr(model, "max_iter"):
            mlflow.log_param("max_iter", model.max_iter)

        for metric_name, value in clf_results[name].items():
            mlflow.log_metric(metric_name, value)

best_clf_name = max(clf_results, key=lambda k: clf_results[k]["f1_macro"])
print(f"Best classifier by macro-F1: {best_clf_name} ({clf_results[best_clf_name]['f1_macro']:.4f})")

# ---------- Regression runs ----------
for name, model in reg_fitted.items():
    with mlflow.start_run(run_name=f"reg_{name.replace(' ', '_')}"):
        mlflow.set_tag("task", "regression")
        mlflow.log_param("model_type", name)
        mlflow.log_param("n_features", X.shape[1])
        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)

        for metric_name, value in reg_results[name].items():
            mlflow.log_metric(metric_name, value)

best_reg_name = min(reg_results, key=lambda k: reg_results[k]["RMSE"])
print(f"Best regressor by RMSE: {best_reg_name} ({reg_results[best_reg_name]['RMSE']:.2f})")

# ---------- Summary of all logged runs ----------
runs_df = mlflow.search_runs()
cols = [c for c in ["tags.mlflow.runName", "tags.task", "start_time"] if c in runs_df.columns]
print("\nLogged runs (stored in ./mlruns):")
print(runs_df[cols])
